In [54]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [55]:
train = pd.read_csv("train.csv")
features = pd.read_csv("features.csv")
stores = pd.read_csv("stores.csv")

print(train.shape)
print(features.shape)
print(stores.shape)

(421570, 5)
(8190, 12)
(45, 3)


In [56]:
data = train.merge(
    features,
    on=["Store","Date","IsHoliday"],
    how="left"
)

data = data.merge(
    stores,
    on="Store",
    how="left"
)

print(data.shape)

data.head()

(421570, 16)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


In [57]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  str    
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
 5   Temperature   421570 non-null  float64
 6   Fuel_Price    421570 non-null  float64
 7   MarkDown1     150681 non-null  float64
 8   MarkDown2     111248 non-null  float64
 9   MarkDown3     137091 non-null  float64
 10  MarkDown4     134967 non-null  float64
 11  MarkDown5     151432 non-null  float64
 12  CPI           421570 non-null  float64
 13  Unemployment  421570 non-null  float64
 14  Type          421570 non-null  str    
 15  Size          421570 non-null  int64  
dtypes: bool(1), float64(10), int64(3), str(2)
memory usage: 48.6 MB


In [58]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
Store,421570.0,22.200546,12.785297,1.000,11.000000,22.00000,33.000000,45.000000
Dept,421570.0,44.260317,30.492054,1.000,18.000000,37.00000,74.000000,99.000000
Weekly_Sales,421570.0,15981.258123,22711.183519,-4988.940,2079.650000,7612.03000,20205.852500,693099.360000
Temperature,421570.0,60.090059,18.447931,-2.060,46.680000,62.09000,74.280000,100.140000
Fuel_Price,421570.0,3.361027,0.458515,2.472,2.933000,3.45200,3.738000,4.468000
MarkDown1,150681.0,7246.420196,8291.221345,0.270,2240.270000,5347.45000,9210.900000,88646.760000
MarkDown2,111248.0,3334.628621,9475.357325,-265.760,41.600000,192.00000,1926.940000,104519.540000
MarkDown3,137091.0,1439.421384,9623.078290,-29.100,5.080000,24.60000,103.990000,141630.610000
MarkDown4,134967.0,3383.168256,6292.384031,0.220,504.220000,1481.31000,3595.040000,67474.850000
MarkDown5,151432.0,4628.975079,5962.887455,135.160,1878.440000,3359.45000,5563.800000,108519.280000


In [59]:
data.isnull().sum().sort_values(ascending=False)

MarkDown2       310322
MarkDown4       286603
MarkDown3       284479
MarkDown1       270889
MarkDown5       270138
Store                0
Date                 0
Dept                 0
Fuel_Price           0
Temperature          0
IsHoliday            0
Weekly_Sales         0
CPI                  0
Unemployment         0
Type                 0
Size                 0
dtype: int64

In [60]:
numeric_cols = data.select_dtypes(include=np.number).columns

for col in numeric_cols:
    data[col] = data[col].fillna(
        data[col].median()
    )

In [61]:
data.isnull().sum().sum()

np.int64(0)

In [62]:
data["Date"] = pd.to_datetime(data["Date"])

In [63]:
data["Year"] = data["Date"].dt.year
data["Month"] = data["Date"].dt.month
data["Week"] = data["Date"].dt.isocalendar().week.astype(int)
data["Quarter"] = data["Date"].dt.quarter
data["Day"] = data["Date"].dt.day

In [64]:
data = pd.get_dummies(
    data,
    columns=["Type"],
    drop_first=True
)

In [65]:
data = data.sort_values("Date")

data.reset_index(
    drop=True,
    inplace=True
)

In [66]:
split_idx = int(len(data)*0.8)

train_df = data.iloc[:split_idx]
test_df = data.iloc[split_idx:]

print(train_df.shape)
print(test_df.shape)

(337256, 22)
(84314, 22)


In [67]:
train_ts = train_df.copy()
test_ts = test_df.copy()

In [68]:
ml_train = train_df.copy()
ml_test = test_df.copy()

ml_train.drop("Date", axis=1, inplace=True)
ml_test.drop("Date", axis=1, inplace=True)

In [69]:
X_train = ml_train.drop(
    "Weekly_Sales",
    axis=1
)

y_train = ml_train["Weekly_Sales"]

X_test = ml_test.drop(
    "Weekly_Sales",
    axis=1
)

y_test = ml_test["Weekly_Sales"]

In [70]:
from sklearn.preprocessing import MinMaxScaler

x_scaler = MinMaxScaler()

X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

In [71]:
y_scaler = MinMaxScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.values.reshape(-1,1)
)

y_test_scaled = y_scaler.transform(
    y_test.values.reshape(-1,1)
)

In [72]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(337256, 20)
(84314, 20)
(337256,)
(84314,)


In [73]:
print(X_train.isnull().sum().sum())
print(X_test.isnull().sum().sum())

0
0


In [74]:
data = data.sort_values("Date")

split_idx = int(len(data)*0.8)

train_df = data.iloc[:split_idx]
test_df = data.iloc[split_idx:]

In [75]:
sales_ts = (
    data.groupby("Date")["Weekly_Sales"]
    .sum()
    .sort_index()
)

In [76]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [77]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

train_size = int(len(sales_ts)*0.8)

train_ts = sales_ts[:train_size]
test_ts = sales_ts[train_size:]

model = SARIMAX(
    train_ts,
    order=(1,1,1),
    seasonal_order=(1,1,1,52)
)

results = model.fit()

pred = results.forecast(len(test_ts))

sarima_mae = mean_absolute_error(test_ts, pred)
sarima_rmse = np.sqrt(mean_squared_error(test_ts, pred))
sarima_r2 = r2_score(test_ts, pred)

c:\Users\hemae\OneDrive\Desktop\ieee\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-FRI will be used.
  self._init_dates(dates, freq)
c:\Users\hemae\OneDrive\Desktop\ieee\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-FRI will be used.
  self._init_dates(dates, freq)


In [78]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

ets = ExponentialSmoothing(
    train_ts,
    trend="add",
    seasonal="add",
    seasonal_periods=52
)

ets_fit = ets.fit()

ets_pred = ets_fit.forecast(len(test_ts))

ets_mae = mean_absolute_error(test_ts, ets_pred)
ets_rmse = np.sqrt(mean_squared_error(test_ts, ets_pred))
ets_r2 = r2_score(test_ts, ets_pred)

c:\Users\hemae\OneDrive\Desktop\ieee\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-FRI will be used.
  self._init_dates(dates, freq)


In [79]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs=1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)



In [80]:
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

In [81]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42,
    n_jobs=2
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)


xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_r2 = r2_score(y_test, xgb_pred)

In [82]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_x.fit_transform(X_train)
X_test_scaled = scaler_x.transform(X_test)

y_train_scaled = scaler_y.fit_transform(
    y_train.values.reshape(-1,1)
)

y_test_scaled = scaler_y.transform(
    y_test.values.reshape(-1,1)
)

In [83]:
X_train_lstm = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    1,
    X_train_scaled.shape[1]
)

X_test_lstm = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    1,
    X_test_scaled.shape[1]
)

In [84]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

lstm = Sequential([
    LSTM(
        50,
        input_shape=(
            X_train_lstm.shape[1],
            X_train_lstm.shape[2]
        )
    ),
    Dense(1)
])

lstm.compile(
    optimizer="adam",
    loss="mse"
)

history = lstm.fit(
    X_train_lstm,
    y_train_scaled,
    epochs=10,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

Epoch 1/10
1186/1186 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 9.4050e-04 - val_loss: 8.5830e-04
Epoch 2/10
1186/1186 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 8.5232e-04 - val_loss: 8.6081e-04
Epoch 3/10
1186/1186 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 8.3961e-04 - val_loss: 8.5826e-04
Epoch 4/10
1186/1186 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 8.2964e-04 - val_loss: 8.0563e-04
Epoch 5/10
1186/1186 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 8.2060e-04 - val_loss: 8.2688e-04
Epoch 6/10
1186/1186 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 8.1640e-04 - val_loss: 8.3254e-04
Epoch 7/10
1186/1186 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 8.1411e-04 - val_loss: 7.9953e-04
Epoch 8/10
1186/1186 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 8.1086e-04 - val_loss: 8.2404e-04
Epoch 9/10
1186/1186 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 8.1030e-04 - val_loss: 7.9849e-04
Epoch 10/10
1186/1186 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 8.0823e-04 - val_loss: 8.2897e-04


In [85]:
lstm_pred_scaled = lstm.predict(X_test_lstm)

lstm_pred = scaler_y.inverse_transform(
    lstm_pred_scaled
)

2635/2635 ━━━━━━━━━━━━━━━━━━━━ 2s 695us/step


In [86]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

lstm_mae = mean_absolute_error(
    y_test,
    lstm_pred
)

lstm_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        lstm_pred
    )
)

lstm_r2 = r2_score(
    y_test,
    lstm_pred
)

print(lstm_mae)
print(lstm_rmse)
print(lstm_r2)

11431.78796718877
19508.447250615503
0.20988029690871768


In [87]:
comparison = pd.DataFrame({
    "Model":[
        "SARIMA",
        "ETS",
        "Random Forest",
        "XGBoost",
        "LSTM"
    ],
    "MAE":[
        sarima_mae,
        ets_mae,
        rf_mae,
        xgb_mae,
        lstm_mae
    ],
    "RMSE":[
        sarima_rmse,
        ets_rmse,
        rf_rmse,
        xgb_rmse,
        lstm_rmse
    ],
    "R2":[
        sarima_r2,
        ets_r2,
        rf_r2,
        xgb_r2,
        lstm_r2
    ]
})

comparison.sort_values(
    "R2",
    ascending=False
)

,Model,MAE,RMSE,R2
2,Random Forest,4.113525e+03,7.123209e+03,0.894659
3,XGBoost,4.626157e+03,7.429388e+03,0.885408
4,LSTM,1.143179e+04,1.950845e+04,0.209880
1,ETS,1.825135e+06,2.192119e+06,-0.604188
0,SARIMA,5.903988e+06,6.035320e+06,-11.159833


In [88]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

param_grid = {
    "n_estimators":[50,100,200],
    "max_depth":[3,5,7],
    "learning_rate":[0.01,0.05,0.1]
}

search = RandomizedSearchCV(
    XGBRegressor(
        random_state=42
    ),
    param_grid,
    n_iter=5,
    cv=3,
    scoring="r2",
    n_jobs=-1
)

search.fit(
    X_train,
    y_train
)

print(search.best_params_)

{'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.1}


In [89]:
from xgboost import XGBRegressor

final_xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

final_xgb_model.fit(X_train, y_train)

y_pred = final_xgb_model.predict(X_test)

In [90]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "n_estimators": [30, 50, 80],
    "max_depth": [5, 10, 15],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

search = RandomizedSearchCV(
    RandomForestRegressor(
        random_state=42,
        n_jobs=1
    ),
    param_distributions=param_grid,
    n_iter=5,
    cv=3,
    scoring="r2",
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print(search.best_params_)
print(search.best_score_)

{'n_estimators': 80, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 15}
0.9179103287093896


In [91]:
from sklearn.ensemble import RandomForestRegressor

final_rf_model = RandomForestRegressor(
    n_estimators=80,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

final_rf_model.fit(X_train, y_train)

y_pred = final_rf_model.predict(X_test)

In [92]:
y_pred_xgb = final_xgb_model.predict(X_test)
r2_xgb = r2_score(y_test, y_pred_xgb)

y_pred_rf = final_rf_model.predict(X_test)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"XGBoost R2 Score: {r2_xgb}")
print(f"Random Forest R2 Score: {r2_rf}")

XGBoost R2 Score: 0.8833011729247319
Random Forest R2 Score: 0.9630353528782518
